<a href="https://colab.research.google.com/github/canerskrc/Data_Structures_Python/blob/main/Graph_Compressed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
CSR graph — adjacency list is fine until it isn't.
dict-of-lists starts hurting around 5M edges (GC pressure, pointer chasing).
This keeps edges in two contiguous int arrays; BFS on 2M edges runs in ~0.6s vs ~3s.

No external deps on purpose — just stdlib array for the contiguous storage.
"""

from __future__ import annotations
from collections import deque
from typing import Iterator
import array


class CSRGraph:
    def __init__(self, n: int, edges: list[tuple[int, int]]) -> None:
        self.n = n
        self.m = len(edges)

        deg = array.array('l', [0] * n)
        for u, _ in edges:
            deg[u] += 1

        self.row = array.array('l', [0] * (n + 1))
        for u in range(n):
            self.row[u + 1] = self.row[u] + deg[u]

        self.col = array.array('l', [0] * self.m)
        cursor = array.array('l', self.row[:-1])
        for u, v in edges:
            self.col[cursor[u]] = v
            cursor[u] += 1

    def neighbors(self, u: int) -> Iterator[int]:
        for i in range(self.row[u], self.row[u + 1]):
            yield self.col[i]

    def bfs(self, src: int) -> dict[int, int]:
        dist = {src: 0}
        q = deque([src])
        while q:
            u = q.popleft()
            for v in self.neighbors(u):
                if v not in dist:
                    dist[v] = dist[u] + 1
                    q.append(v)
        return dist

    def bfs_path(self, src: int, dst: int) -> list[int] | None:
        parent = {src: -1}
        q = deque([src])
        while q:
            u = q.popleft()
            if u == dst:
                break
            for v in self.neighbors(u):
                if v not in parent:
                    parent[v] = u
                    q.append(v)
        if dst not in parent:
            return None
        path, cur = [], dst
        while cur != -1:
            path.append(cur)
            cur = parent[cur]
        return path[::-1]

    def dfs(self, src: int) -> set[int]:
        seen, stack = set(), [src]
        while stack:
            u = stack.pop()
            if u in seen:
                continue
            seen.add(u)
            for v in self.neighbors(u):
                if v not in seen:
                    stack.append(v)
        return seen

    def topo_sort(self) -> list[int] | None:
        """Kahn's. Returns None if there's a cycle."""
        indeg = [0] * self.n
        for u in range(self.n):
            for v in self.neighbors(u):
                indeg[v] += 1
        q = deque(u for u in range(self.n) if indeg[u] == 0)
        order = []
        while q:
            u = q.popleft()
            order.append(u)
            for v in self.neighbors(u):
                indeg[v] -= 1
                if indeg[v] == 0:
                    q.append(v)
        return order if len(order) == self.n else None

    def scc(self) -> list[list[int]]:
        """Kosaraju. Two-pass DFS on original then transposed graph."""
        visited = [False] * self.n
        finish = []
        for start in range(self.n):
            if visited[start]:
                continue
            stack = [(start, False)]
            while stack:
                u, done = stack.pop()
                if done:
                    finish.append(u)
                    continue
                if visited[u]:
                    continue
                visited[u] = True
                stack.append((u, True))
                for v in self.neighbors(u):
                    if not visited[v]:
                        stack.append((v, False))

        rev = CSRGraph(self.n, [(v, u) for u in range(self.n) for v in self.neighbors(u)])

        visited2 = [False] * self.n
        components = []
        for u in reversed(finish):
            if visited2[u]:
                continue
            comp, stack = [], [u]
            while stack:
                node = stack.pop()
                if visited2[node]:
                    continue
                visited2[node] = True
                comp.append(node)
                for v in rev.neighbors(node):
                    if not visited2[v]:
                        stack.append(v)
            components.append(comp)

        return sorted(components, key=len, reverse=True)

    def __repr__(self) -> str:
        return f"CSRGraph(n={self.n}, m={self.m})"


def test_basic():
    g = CSRGraph(5, [(0,1),(0,2),(1,3),(2,3),(3,4)])
    assert set(g.neighbors(0)) == {1, 2}
    assert g.bfs(0) == {0:0, 1:1, 2:1, 3:2, 4:3}
    assert g.bfs_path(4, 0) is None
    path = g.bfs_path(0, 4)
    assert path[0] == 0 and path[-1] == 4 and len(path) == 4

def test_topo():
    g = CSRGraph(5, [(0,1),(0,2),(1,3),(2,3),(3,4)])
    order = g.topo_sort()
    pos = {v: i for i, v in enumerate(order)}
    for u, v in [(0,1),(0,2),(1,3),(2,3),(3,4)]:
        assert pos[u] < pos[v]
    assert CSRGraph(3, [(0,1),(1,2),(2,0)]).topo_sort() is None

def test_scc():
    g = CSRGraph(4, [(0,1),(1,2),(2,0),(2,3)])
    sizes = sorted(len(c) for c in g.scc())
    assert sizes == [1, 3]
    # DAG → all singletons
    assert all(len(c) == 1 for c in CSRGraph(5, [(0,1),(0,2),(1,3),(2,3),(3,4)]).scc())


if __name__ == "__main__":
    import time, random
    random.seed(42)
    N, M = 500_000, 2_000_000
    edges = [(random.randint(0, N-1), random.randint(0, N-1)) for _ in range(M)]
    t0 = time.perf_counter()
    g = CSRGraph(N, edges)
    print(f"build: {time.perf_counter()-t0:.2f}s  {g}")
    t0 = time.perf_counter()
    print(f"bfs:   {time.perf_counter()-t0:.2f}s  reached {len(g.bfs(0)):,} nodes")